# DDL CPTEC Obs (MERGE + SAMeT)

Crea el Volume, las tablas Bronze (grilla recortada al bounding box de la cuenca) y las tablas
Silver (agregados por sub-cuenca) de las observaciones en grilla de CPTEC/INPE: **MERGE**
(precipitacion diaria 0,1 grado, satelite GPM-IMERG + pluviometros) y **SAMeT** (temperatura
diaria TMAX/TMED/TMIN 0,05 grado, observaciones + ERA5). Siembra `weather.silver.grid_subcuenca`
(punto de grilla -> sub-cuenca) desde el catalogo generado en local por
`notebooks_local/cptec_obs/build_grid_subcuenca.py`. Ver `docs/data_sources.md` §9.6/§9.7 y
Decision 033. Las columnas nuevas de Gold viven en `DDL_Silver_Gold.ipynb`, como el resto de Gold.

In [ ]:
spark.sql('CREATE VOLUME IF NOT EXISTS weather.raw.cptec_volume')

for sub in ['merge/daily', 'samet/daily', 'staging', 'catalogo']:
    dbutils.fs.mkdirs(f'/Volumes/weather/raw/cptec_volume/{sub}')

In [ ]:
# Bronze: un registro por (fecha, punto de grilla). Se conserva lo que publica CPTEC dentro del
# bounding box (5.244 puntos MERGE / ~20.300 puntos SAMeT por dia); la asignacion a sub-cuenca es
# de Silver. `source_last_modified` es el Last-Modified HTTP del archivo de origen: MERGE se
# regenera en los primeros dias del mes siguiente y SAMeT ~7 dias despues (ERA5), y Bronze
# actualiza la fila cuando llega una version mas nueva (ver ETL_Bronze_CPTEC_Obs.ipynb).
# CLUSTER BY (fecha): el MERGE incremental filtra por rango de fechas y asi poda archivos.
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.merge_precip_grid (
  fecha DATE,
  latitude DOUBLE,
  longitude DOUBLE,
  prec_mm DOUBLE,
  nest INT,
  source_file STRING,
  source_last_modified TIMESTAMP,
  source_api STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
CLUSTER BY (fecha)
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.samet_temp_grid (
  fecha DATE,
  latitude DOUBLE,
  longitude DOUBLE,
  tmed_c DOUBLE,
  tmax_c DOUBLE,
  tmin_c DOUBLE,
  nobs_tmed INT,
  nobs_tmax INT,
  nobs_tmin INT,
  source_file STRING,
  source_last_modified TIMESTAMP,
  source_api STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
CLUSTER BY (fecha)
''')

In [ ]:
# Silver: agregado por (fecha, sub-cuenca, fuente). `cobertura_pct` = puntos con dato / puntos
# de la sub-cuenca en grid_subcuenca (R8, Decision 019: cobertura expuesta, sin porton).
# `es_preliminar` marca las filas construidas con la version preliminar del archivo de origen
# (antes de la regeneracion de CPTEC) -- se recalcula cuando Bronze recibe la version final.
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.grid_subcuenca (
  grilla STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  subcuenca STRING,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.precip_grid_daily (
  fecha DATE,
  subcuenca STRING,
  fuente STRING,
  prec_media_mm DOUBLE,
  prec_max_mm DOUBLE,
  puntos_grilla BIGINT,
  puntos_esperados BIGINT,
  cobertura_pct DOUBLE,
  puntos_con_pluviometro BIGINT,
  pluviometros BIGINT,
  source_last_modified TIMESTAMP,
  es_preliminar BOOLEAN,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.temp_grid_daily (
  fecha DATE,
  subcuenca STRING,
  fuente STRING,
  temp_media_c DOUBLE,
  temp_max_c DOUBLE,
  temp_min_c DOUBLE,
  temp_max_abs_c DOUBLE,
  temp_min_abs_c DOUBLE,
  puntos_grilla BIGINT,
  puntos_esperados BIGINT,
  cobertura_pct DOUBLE,
  nobs_total BIGINT,
  source_last_modified TIMESTAMP,
  es_preliminar BOOLEAN,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

In [ ]:
# Siembra weather.silver.grid_subcuenca desde el catalogo generado en local
# (notebooks_local/cptec_obs/build_grid_subcuenca.py: union espacial de los centros de celda de
# cada grilla contra SIG/subcuencas_modelo.geojson, subido con sync_to_databricks.py --catalogo).
# Mismo criterio que la siembra de estacion_subcuenca desde los inventarios ANA/INMET: la
# geometria se resuelve en local con geopandas, Databricks solo hace el join por clave.
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CATALOGO_PATH = '/Volumes/weather/raw/cptec_volume/catalogo/grid_subcuenca.json'

try:
    dbutils.fs.ls(CATALOGO_PATH)
    catalogo_existe = True
except Exception:
    catalogo_existe = False

if not catalogo_existe:
    print(f'{CATALOGO_PATH} no existe todavia: correr sync_to_databricks.py --catalogo. grid_subcuenca queda como esta.')
else:
    grid = (
        spark.read.option('multiLine', True).json(CATALOGO_PATH)
        .select(
            F.col('grilla').cast('string').alias('grilla'),
            F.round(F.col('latitude').cast('double'), 3).alias('latitude'),
            F.round(F.col('longitude').cast('double'), 3).alias('longitude'),
            F.col('subcuenca').cast('string').alias('subcuenca'),
        )
        .filter(F.col('grilla').isNotNull() & F.col('subcuenca').isNotNull())
        .dropDuplicates(['grilla', 'latitude', 'longitude'])
        .withColumn('updated_at', F.current_timestamp())
    )
    (
        DeltaTable.forName(spark, 'weather.silver.grid_subcuenca').alias('t')
        .merge(grid.alias('s'), 't.grilla = s.grilla AND t.latitude = s.latitude AND t.longitude = s.longitude')
        .whenMatchedUpdate(set={'subcuenca': 's.subcuenca', 'updated_at': 's.updated_at'})
        .whenNotMatchedInsertAll()
        .execute()
    )
    spark.table('weather.silver.grid_subcuenca').groupBy('grilla', 'subcuenca').count().orderBy('grilla', 'subcuenca').show()